# Una gramática: LangGraph + Agentic Systems · 2.1.4

Se declaran **state, nodes y edges** con LangGraph. Agent organiza operaciones; System coordina unidades; Environment ejecuta su grafo por observación. Todos usan `graph=Graph(...)`. `compile()` valida y construye; `run()` ejecuta y devuelve RunResult.

Este notebook es ejecutable offline: no hace llamadas a modelos ni arranca un servidor. Requiere instalar el wheel local 2.1.4. Las pruebas live de dev1 son históricas y no certifican este artefacto nuevo.

En paralelo, cada rama escribe sus propios campos. Una arista desde una lista de nodos es una barrera: el siguiente nodo espera a todos. No hay otro constructor de composición ni un callback de combinación del toolkit.

In [2]:
import pydantic
from pathlib import Path
import sys
import agentic_systems as toolkit

assert toolkit.__version__ == '2.1.4'
ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples/common_graph/application.py").is_file()
    )
)
sys.path.insert(0, str(ROOT))


class State(pydantic.BaseModel):
    value: int
    increased: int | None = None
    doubled: int | None = None


def increase(state):
    return {"value": state.value + 1}


def double(state):
    return {"value": state.value * 2}

## Agent: dos operaciones en orden

In [3]:
graph = toolkit.Graph(State)
graph.add_node("increase", increase)
graph.add_node("double", double)
graph.add_edge(toolkit.START, "increase")
graph.add_edge("increase", "double")
graph.add_edge("double", toolkit.END)
agent = toolkit.Agent("arithmetic", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

arithmetic: COMPLETED
Execution: run-c8af5490659d4a31bdbb5e6e0df70786
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Step double: completed


## System: coordinar dos Agents

Cada agente tiene su propio grafo y se usa directamente como nodo del sistema.

In [4]:
increase_graph = toolkit.Graph(State)
increase_graph.add_node("increase", increase)
increase_graph.add_edge(toolkit.START, "increase")
increase_graph.add_edge("increase", toolkit.END)
incrementer = toolkit.Agent("incrementer", graph=increase_graph)
double_graph = toolkit.Graph(State)
double_graph.add_node("double", double)
double_graph.add_edge(toolkit.START, "double")
double_graph.add_edge("double", toolkit.END)
multiplier = toolkit.Agent("multiplier", graph=double_graph)
graph = toolkit.Graph(State)
graph.add_node("incrementer", incrementer)
graph.add_node("multiplier", multiplier)
graph.add_edge(toolkit.START, "incrementer")
graph.add_edge("incrementer", "multiplier")
graph.add_edge("multiplier", toolkit.END)
system = toolkit.System("team", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

team: COMPLETED
Execution: run-3bf546f3462942688bf4db54765c04cd
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Agent multiplier: completed
- Step double: completed


## Agent: ramas concurrentes

Ambas leen value=2. Una escribe increased=3 y otra doubled=4. La barrera habilita total cuando ambas terminaron. Cada nodo devuelve solo sus actualizaciones.

In [5]:
def left(state):
    return {"increased": state.value + 1}


def right(state):
    return {"doubled": state.value * 2}


def total(state):
    return {"value": state.increased + state.doubled}


graph = toolkit.Graph(State)
graph.add_node("left", left)
graph.add_node("right", right)
graph.add_node("total", total)
graph.add_edge(toolkit.START, "left")
graph.add_edge(toolkit.START, "right")
graph.add_edge(["left", "right"], "total")
graph.add_edge("total", toolkit.END)
agent = toolkit.Agent("concurrent", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 7
assert result.check_invariants().ok
print(result.human_results())

concurrent: COMPLETED
Execution: run-bd2125e3f0c8489ca4384caa52443f2e
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Step left: completed
- Step right: completed
- Step total: completed


## System: Agents concurrentes

Es el mismo mecanismo de LangGraph. Los agentes usados como nodos propagan sus campos escritos, sin reenviar todos los campos recibidos.

In [6]:
left_graph = toolkit.Graph(State)
left_graph.add_node("left", left)
left_graph.add_edge(toolkit.START, "left")
left_graph.add_edge("left", toolkit.END)
left_agent = toolkit.Agent("left", graph=left_graph)
right_graph = toolkit.Graph(State)
right_graph.add_node("right", right)
right_graph.add_edge(toolkit.START, "right")
right_graph.add_edge("right", toolkit.END)
right_agent = toolkit.Agent("right", graph=right_graph)
graph = toolkit.Graph(State)
graph.add_node("left", left_agent)
graph.add_node("right", right_agent)
graph.add_node("total", total)
graph.add_edge(toolkit.START, "left")
graph.add_edge(toolkit.START, "right")
graph.add_edge(["left", "right"], "total")
graph.add_edge("total", toolkit.END)
system = toolkit.System("concurrent-team", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 7
assert result.check_invariants().ok
print(result.human_results())

concurrent-team: COMPLETED
Execution: run-14128a30bc284071b6976dbe78320774
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Agent left: completed
- Step left: completed
- Agent right: completed
- Step right: completed
- Step total: completed


## Agent: decisión

La función de ruta devuelve un nombre de nodo o END; no hay un objeto de rutas adicional.

In [7]:
graph = toolkit.Graph(State)
graph.add_node("increase", increase)
graph.add_node("double", double)
graph.add_edge(toolkit.START, "increase")
graph.add_conditional_edges(
    "increase",
    lambda state: "double" if state.value >= 0 else toolkit.END,
    ["double", toolkit.END],
)
graph.add_edge("double", toolkit.END)
agent = toolkit.Agent("decision", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

decision: COMPLETED
Execution: run-9600e71909954f59a1540d60c04ef4bb
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Decision increase -> "double"
- Step double: completed


## System: decisión

La función de ruta devuelve un nombre de nodo o END; no hay un objeto de rutas adicional.

In [8]:
graph = toolkit.Graph(State)
graph.add_node("increase", incrementer)
graph.add_node("double", multiplier)
graph.add_edge(toolkit.START, "increase")
graph.add_conditional_edges(
    "increase",
    lambda state: "double" if state.value >= 0 else toolkit.END,
    ["double", toolkit.END],
)
graph.add_edge("double", toolkit.END)
system = toolkit.System("decision", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

decision: COMPLETED
Execution: run-eeb4a30b99cb44dda6d827ed671774c6
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Decision increase -> "double"
- Agent multiplier: completed
- Step double: completed


## Context: información de dominio, separada del state

`Context(data, sources=...)` es un snapshot JSON inmutable. Agent, System y Environment aceptan `context=`; sus hijos lo heredan. Una clave en conflicto se rechaza, sin sustituirla silenciosamente.

- `context.data`: reglas, vocabulario y datos del universo de trabajo.
- `context.sources`: identificadores de procedencia.
- `context.episodes`: episodios seleccionados de Memory al entrar en la unidad.
- `context.lineage`: hasta ocho resúmenes compactos de evidencia previa. Son observaciones, no autoridad.

Una Tool lo recibe mediante `*, context: toolkit.Context`. No aparece como argumento que deba inventar el modelo. Skills agrupan capacidades e instrucciones; Context aporta referencia informacional. El state conserva los datos necesarios para calcular. La política no se copia automáticamente dentro del state.

In [9]:
domain = toolkit.Context(
    {"currency": "MXN", "tax_percent": 16}, sources=("tax-policy-v1",)
)
changed_copy = domain.data
changed_copy["tax_percent"] = 99
assert domain.data["tax_percent"] == 16
print("Context:", domain.to_dict())
print("Fingerprint:", domain.fingerprint)

Context: {'data': {'currency': 'MXN', 'tax_percent': 16}, 'sources': ['tax-policy-v1'], 'episodes': [], 'lineage': []}
Fingerprint: 9458b62c01ac3d63c4695cd67069372d6c4372e74b029d40201194abb01402af


## load_skill: cargar una capacidad y usarla en un grafo

El paquete contiene SKILL.md, skill.py y tools.py. `build_skill()` declara la capacidad runtime; SKILL.md la documenta. Se ejecuta código Python local al cargarlo, por lo que el origen debe ser conocido.

Input 100 MXN → cargar Skill → Tool quote consulta Context → output 116 MXN. Memory aporta el episodio anterior en la segunda ejecución.

In [10]:
class QuoteState(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    policy_source: str | None = None
    previous_episodes: int | None = None
    preceding_events: int | None = None


capability = toolkit.load_skill(ROOT / "examples/common_graph/skills/pricing")
graph = toolkit.Graph(QuoteState)
graph.add_node("quote", capability.tools[0])
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
quoter = toolkit.Agent(
    "quote", graph=graph, skills=(capability,), context=domain, memory=toolkit.Memory()
)
first_quote = quoter.run({"subtotal": 100})
second_quote = quoter.run({"subtotal": 100})
assert first_quote.ok and first_quote.data["total"] == 116
assert second_quote.data["previous_episodes"] == 1
print("Input:", {"subtotal": 100})
print("Proceso: load_skill → quote(Context) → Memory")
print("Output:", first_quote.data)
print(
    "Episodios previos en la segunda ejecución:", second_quote.data["previous_episodes"]
)

Input: {'subtotal': 100}
Proceso: load_skill → quote(Context) → Memory
Output: {'subtotal': 100, 'total': 116, 'currency': 'MXN', 'policy_source': 'tax-policy-v1', 'previous_episodes': 0, 'preceding_events': 0}
Episodios previos en la segunda ejecución: 1


### El mismo Context en Tools y nodos

Ambos reciben `*, context: toolkit.Context`: el snapshot efectivo de la unidad, con dominio, fuentes, Memory seleccionada y evidencia precedente. No se copia Context dentro de state ni se importa Runtime.

In [11]:
def quote_from_reference(state, *, context: toolkit.Context):
    domain = context
    return {
        "total": state.subtotal * (100 + domain.data["tax_percent"]) // 100,
        "currency": domain.data["currency"],
    }


graph = toolkit.Graph(QuoteState)
graph.add_node("quote", quote_from_reference)
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
contextual_result = toolkit.Agent("runtime-context", graph=graph, context=domain).run(
    {"subtotal": 100}
)
assert contextual_result.ok and contextual_result.data["total"] == 116
print("Input:", {"subtotal": 100}, "→ Runtime.context →", contextual_result.data)

Input: {'subtotal': 100} → Runtime.context → {'subtotal': 100, 'total': 116, 'currency': 'MXN'}


## Proyectar una entrada

Las Tools pueden ser nodos directamente. Cuando el input tiene otra forma, una función nodo selecciona los campos y llama a Tool.run. La proyección usa model_dump(exclude_unset=True) para conservar ausencias y dejar que el contrato rechace campos requeridos faltantes.

In [12]:
class NestedState(pydantic.BaseModel):
    order: QuoteState
    total: int | None = None
    currency: str | None = None
    policy_source: str | None = None
    previous_episodes: int | None = None
    preceding_events: int | None = None


graph = toolkit.Graph(NestedState)
def quote_step(state):
    return capability.tools[0].run(state.order.model_dump(mode="json", exclude_unset=True))
graph.add_node(
    "quote",
    quote_step,
)
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
projected = toolkit.Agent("nested-input", graph=graph, context=domain).run(
    {"order": {"subtotal": 100}}
)
assert projected.ok and projected.data["total"] == 116
print("Input:", {"order": {"subtotal": 100}}, "→ proyección →", projected.data)

Input: {'order': {'subtotal': 100}} → proyección → {'order': {'subtotal': 100}, 'total': 116, 'currency': 'MXN', 'policy_source': 'tax-policy-v1', 'previous_episodes': 0, 'preceding_events': 0}


## Tools, Skills, Agents, System, Environment y Eval juntos

R-17 cuesta 240 MXN y contiene dos componentes. Falta uno. Un clasificador opera sobre evidencia estructurada; herramientas separadas calculan precio y política; una decisión solicita revisión; el programa pausa y se reanuda con aprobación. Environment procesa dos pedidos; Eval verifica el contrato.

Todo está construido con Graph. Los tres SDK se prueban aquí con Python, no con modelos simulados. Sus bucles internos siguen siendo responsabilidad del SDK del actor; LangGraph coordina sus nodos.

In [13]:
"""A refund workflow: Python operators, optional real language interpretation."""
import pydantic
import agentic_systems as toolkit

CASE = {
    "order_id": " R-17 ",
    "amount": 240,
    "expected_items": ["adapter", "sensor"],
    "received_items": ["adapter"],
    "customer_note": "La caja llegó sellada. Encontré el adaptador; el sensor del albarán no venía. Quiero devolver todo.",
}
SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "order_id",
        "amount",
        "total_units",
        "missing_units",
        "reason",
        "sources",
    ],
    "properties": {
        "order_id": {"type": "string"},
        "amount": {"type": "integer"},
        "total_units": {"type": "integer", "minimum": 1},
        "missing_units": {"type": "integer", "minimum": 0},
        "reason": {"type": "string", "minLength": 5},
        "sources": {"type": "array", "items": {"type": "string"}, "minItems": 1},
    },
}


@toolkit.tool
def normalize(
    order_id: str,
    amount: int,
    expected_items: list,
    received_items: list,
    customer_note: str,
) -> dict:
    """Normalize identifiers and preserve the evidence, without interpretation."""
    if amount < 0 or not expected_items:
        raise ValueError("A case needs nonnegative amount and an itemized packing slip")
    return {
        "order_id": order_id.strip(),
        "amount": amount,
        "expected_items": expected_items,
        "received_items": received_items,
        "customer_note": customer_note,
    }


@toolkit.tool
def classify(
    order_id: str,
    amount: int,
    expected_items: list,
    received_items: list,
    customer_note: str,
) -> dict:
    """Deterministic operator for structured item evidence; no language understanding."""
    return {
        "order_id": order_id,
        "amount": amount,
        "total_units": len(expected_items),
        "missing_units": len(set(expected_items) - set(received_items)),
        "reason": "Difference between packing slip and received item list",
        "sources": ["packing-slip", "customer-report"],
    }


@toolkit.tool
def price(
    order_id: str,
    amount: int,
    total_units: int,
    missing_units: int,
    reason: str,
    sources: list,
) -> dict:
    """Compute money in integer units; never delegate arithmetic to a model."""
    if not 0 <= missing_units <= total_units or total_units < 1:
        raise ValueError("Inconsistent quantities")
    return {"order_id": order_id, "refund": amount * missing_units // total_units}


@toolkit.tool
def policy(
    order_id: str,
    amount: int,
    total_units: int,
    missing_units: int,
    reason: str,
    sources: list,
    *,
    context: toolkit.Context,
) -> dict:
    """Use semantic domain Context and retain the classifier's evidence."""
    return {
        "currency": context.data["currency"],
        "review_threshold": context.data["review_threshold"],
        "reason": reason,
        "sources": sources,
        "missing_units": missing_units,
        "previous_episodes": len(context.episodes),
    }


class RefundState(pydantic.BaseModel):
    order_id: str | None = None
    amount: int | None = None
    expected_items: list | None = None
    received_items: list | None = None
    customer_note: str | None = None
    total_units: int | None = None
    missing_units: int | None = None
    reason: str | None = None
    sources: list | None = None
    refund: int | None = None
    currency: str | None = None
    review_threshold: int | None = None
    previous_episodes: int | None = None
    needs_review: bool | None = None
    status: str | None = None


class RefundOutput(pydantic.BaseModel):
    order_id: str
    refund: int
    currency: str
    missing_units: int
    reason: str
    sources: list
    previous_episodes: int
    status: str


def tool_agent(name, operation, *, skills=()):
    graph = toolkit.Graph(RefundState)
    graph.add_node(name, operation)
    graph.add_edge(toolkit.START, name)
    graph.add_edge(name, toolkit.END)
    return toolkit.Agent(name, graph=graph, skills=skills)


def build_system(framework="native", *, reasoner=None, interactive=False):
    domain = toolkit.Context(
        {
            "currency": "MXN",
            "review_threshold": 100,
            "refund_policy": "Refund only missing items proportionally; the customer's demand is not proof of a fully missing order.",
        },
        sources=("refund-policy-v1", "packing-slip", "customer-report"),
    )
    capability = toolkit.skill(name="refunds", tools=(normalize, price, policy))
    normalizer = tool_agent("normalize", normalize, skills=(capability,))
    classifier = toolkit.Agent(
        "classify",
        runtime=reasoner or toolkit.runtime(provider="python-runtime"),
        framework=toolkit.framework(framework),
        skills=()
        if reasoner
        else (toolkit.skill(name="classification", tools=(classify,)),),
        instructions="Interpret the customer note against packing slip and received items. Apply reference policy. Return only JSON: order_id, amount, total_units, missing_units, short evidence-based reason, sources including packing-slip and customer-report. Copy original amount unchanged; Python calculates refunds."
        if reasoner
        else "",
        contract={"output_schema": SCHEMA},
    )

    def review(data):
        if not interactive:
            return {"status": "pending_review"}
        approved = toolkit.pause(
            {
                "question": "Approve partial refund?",
                "refund": data.refund,
                "order_id": data.order_id,
            }
        )
        if type(approved) is not bool:
            raise TypeError("Approval must be bool")
        return {"status": "approved" if approved else "rejected"}

    graph = toolkit.Graph(RefundState, output_schema=RefundOutput)
    graph.add_node("normalize", normalizer)
    graph.add_node("classify", classifier)
    graph.add_node("price", tool_agent("price", price))
    graph.add_node("policy", tool_agent("policy", policy))
    def decision_step(state):
        return {"needs_review": state.refund > state.review_threshold}
    graph.add_node(
        "decision",
        decision_step,
    )
    graph.add_node("review", review)
    def approve_step(state):
        return {"status": "approved"}
    graph.add_node("approve", approve_step)
    graph.add_edge(toolkit.START, "normalize")
    graph.add_edge("normalize", "classify")
    graph.add_edge("classify", "price")
    graph.add_edge("classify", "policy")
    graph.add_edge(["price", "policy"], "decision")
    graph.add_conditional_edges(
        "decision",
        lambda state: "review" if state.needs_review else "approve",
        ["review", "approve"],
    )
    graph.add_edge("review", toolkit.END)
    graph.add_edge("approve", toolkit.END)
    return toolkit.System(
        "refunds", graph=graph, context=domain, memory=toolkit.Memory()
    )


def check_result(output):
    return (
        output.ok
        and output.data["order_id"] == "R-17"
        and (output.data["refund"] == 120)
        and (output.data["currency"] == "MXN")
        and (output.data["missing_units"] == 1)
        and ({"packing-slip", "customer-report"} <= set(output.data["sources"]))
        and output.check_invariants().ok
    )


def demonstrate(framework="native"):
    definition = build_system(framework, interactive=True)
    program = definition.compile()
    waiting = program.run(CASE, timeout_s=40)
    assert waiting.status == "paused", waiting.to_dict()
    final = program.resume(waiting, True)
    assert check_result(final), final.to_dict()
    graph = toolkit.Graph(RefundState, output_schema=RefundOutput)
    graph.add_node("refund", build_system(framework))
    graph.add_edge(toolkit.START, "refund")
    graph.add_edge("refund", toolkit.END)
    episode = toolkit.Environment(
        "daily-refunds",
        graph=graph,
        records=(CASE, CASE),
        state={"total": 0},
        update=lambda state, row, result: {
            "total": state["total"] + result.data["refund"]
        },
    ).run(timeout_s=40)
    evaluation = toolkit.Eval(
        "refund-contract",
        build_system(framework),
        ({"input": CASE, "expected": {"ok": True}},),
        checks=(lambda result, case: {"ok": check_result(result)},),
    ).run(timeout_s=40)
    assert episode.ok and episode.data["summary"]["state"] == {"total": 240}, (
        episode.to_dict()
    )
    assert evaluation.ok, evaluation.to_dict()
    restored = toolkit.RunResult(**final.to_dict())
    assert restored.check_invariants().ok
    return {
        "orchestration": "langgraph",
        "framework": framework,
        "provider": "python-runtime",
        "input": CASE,
        "process": [
            "normalize",
            "classify",
            "price + policy",
            "decision",
            "pause/resume",
        ],
        "output": final.data,
        "run_results": [
            waiting.to_dict(),
            final.to_dict(),
            episode.to_dict(),
            evaluation.to_dict(),
        ],
        "human_lineage": final.lineage(name="Refund R-17").render(),
        "agent_lineage": final.lineage().to_prompt_context(),
        "memory": definition.memory.read(),
    }


reports = [demonstrate(name) for name in ("native", "openai-agents", "strands")]
for report in reports:
    print(report["framework"], report["output"])

native {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
openai-agents {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
strands {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}


## Environment: el mismo parámetro graph

El grafo de cada observación contiene un System. Environment añade su propio grafo temporal observe → run → update → observe/END. `state` es el estado del episodio; Memory son observaciones retenidas entre ejecuciones; Context es la referencia de dominio.

In [14]:
graph = toolkit.Graph(QuoteState)
graph.add_node("quoter", quoter)
graph.add_edge(toolkit.START, "quoter")
graph.add_edge("quoter", toolkit.END)
episode = toolkit.Environment(
    "quotes",
    graph=graph,
    records=({"subtotal": 100}, {"subtotal": 200}),
    context=domain,
    state={"sum": 0},
    update=lambda state, row, result: {"sum": state["sum"] + result.data["total"]},
)
episode_result = episode.run()
assert episode_result.ok and episode_result.data["summary"]["state"] == {"sum": 348}
print(
    "Input: dos presupuestos → grafo por observación →", episode_result.data["summary"]
)
evaluation = toolkit.Eval(
    "tax-contract",
    quoter,
    cases=({"input": {"subtotal": 100}, "expected": {"ok": True}},),
    checks=(lambda result, case: {"ok": result.data["total"] == 116},),
).run()
assert evaluation.ok
print("Eval:", evaluation.data)

Input: dos presupuestos → grafo por observación → {'state': {'sum': 348}, 'processed': 2, 'terminated': True, 'truncated': False, 'reason': 'records_exhausted'}


Eval: {'total': 1, 'passed': 1, 'failed': 0, 'cases': [{'name': '0', 'ok': True, 'input': {'subtotal': 100}, 'execution_id': 'run-ed7383195f8342b9976d097bf9f0a128', 'checks': [{'ok': True, 'path': 'ok'}, {'ok': True}], 'judge': None}]}


## Pausa, linaje y estado compacto

El state contiene datos de trabajo y hasta 32 referencias recientes en el canal técnico `_lineage`. Los eventos completos viven fuera de los checkpoints, en un almacén de evidencia plano por ejecución. La salida final se materializa como RunResult con su árbol de linaje; las filas de Eval e historia del Environment apuntan por ID, sin insertar RunResult completos en state.

Eso evita duplicar árboles en cada transición. No garantiza tamaño constante de los datos de negocio que el usuario decida acumular, ni almacenamiento infinito: la evidencia externa crece con el trabajo y existe un plazo raíz opcional.

Las continuaciones son locales al proceso. Una función interrumpida comienza otra vez al reanudar; separa efectos previos en nodos anteriores o hazlos idempotentes.

In [15]:
program = build_system(interactive=True).compile()
waiting = program.run(CASE, timeout_s=40)
assert waiting.status == "paused"
print("Pregunta:", waiting.interrupts)
restored = toolkit.RunResult(**waiting.to_dict())
finished = program.resume(restored, True)
assert finished.ok and finished.data["refund"] == 120
print("Output:", finished.data)
print(finished.lineage(name="R-17").render())
print("Para otro agente:", finished.lineage().to_prompt_context())

Pregunta: ({'id': '82fe5a3c5adf8d2cc063bddb1dc47a6e', 'value': {'question': 'Approve partial refund?', 'refund': 120, 'order_id': 'R-17'}},)


Output: {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
R-17: OK
{"order_id": "R-17", "refund": 120, "currency": "MXN", "missing_units": 1, "reason": "Difference between packing slip and received item list", "sources": ["packing-slip", "customer-report"], "previous_episodes": 0, "status": "approved"}
- refunds: completed [run-efbe41f3d22f412380543b4dcefb6ba3]
- normalize: completed [run-b61cb8bc2fa64e0e9d96cdb232c0ed73]
- normalize: completed [tool-22d67f05235941118dd5c75d2656e554]
- classify: classify -> {"order_id": "R-17", "amount": 240, "total_units": 2, "missing_units": 1, "reason": "Difference between packing slip and received item list", "sources": ["packing-slip", "customer-report"] [run-ca401f8a71ad4646bcab13bd10787f47]
- eval: classify -> {"order_id": "R-17", "amount": 240, "total_units": 2

## model_artifact: identidad y procedencia

Declara qué pesos se servirían. No descarga, entrena, carga ni valida esos pesos. La revisión main es una referencia mutable; antes de una validación real habría que fijar revisión y hashes.

In [16]:
artifact = toolkit.model_artifact(
    "Qwen/Qwen2.5-0.5B-Instruct",
    revision="main",
    metadata={"purpose": "offline serving declaration; not validated weights"},
)
print(artifact.inspect())

{'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'base_model': None, 'adapter_path': None, 'tokenizer': None, 'revision': 'main', 'quantization': None, 'metadata': {'purpose': 'offline serving declaration; not validated weights'}}


## model_server: plan de serving, sin arrancarlo

Construye configuración, comando y endpoint. En esta ejecución offline se valida el plan y la ocultación de la clave; no se llama start ni health. Los tests separados simulan start/health/stop sin procesos ni red. El arranque real necesita vLLM, pesos y un entorno GPU compatible y queda pendiente.

In [17]:
server = toolkit.model_server(
    artifact,
    profile="custom",
    served_model_name="quote-model",
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    max_num_seqs=1,
    enable_auto_tool_choice=False,
    tool_call_parser=None,
    api_key="example-local-token",
)
connection = toolkit.runtime(
    provider="vllm-runtime",
    model=server.endpoint.model,
    endpoint=server.endpoint.url,
    api_key="example-local-token",
    max_tokens=128,
)
assert server.endpoint.owned is False
assert "example-local-token" not in str(server.inspect())
print("Comando redactado:", server.command)
print("Endpoint previsto:", server.endpoint)
print("Conexión prevista, sin inferencia:", connection.inspect())

Comando redactado: ('vllm', 'serve', 'Qwen/Qwen2.5-0.5B-Instruct', '--host', '127.0.0.1', '--port', '8000', '--served-model-name', 'quote-model', '--gpu-memory-utilization', '0.5', '--max-model-len', '2048', '--max-num-seqs', '1', '--generation-config', 'vllm', '--api-key', '[REDACTED]')
Endpoint previsto: _Endpoint(url='http://127.0.0.1:8000/v1', model='quote-model', backend='vllm', api_key_configured=True, pid=None, owned=False)
Conexión prevista, sin inferencia: {'provider': 'vllm-runtime', 'model': 'quote-model', 'region': None, 'endpoint': 'http://127.0.0.1:8000/v1', 'api_key_configured': True, 'session_configured': False, 'authentication_mode': 'provider-configuration', 'timeout_s': 60.0, 'max_retries': 0, 'max_tool_calls': 5, 'max_turns': 6, 'max_tokens': 128, 'backoff_s': 0.0, 'max_concurrency': 1}


### Solo en el entorno GPU preparado; no ejecutado aquí

```python
server.start()
try:
    assert server.health().status == "healthy"
    # connection ya apunta al endpoint y modelo declarados.
    # Probar inferencia exige una campaña y presupuesto explícitos.
finally:
    server.stop()
```

Esta revisión es offline y no está publicada. Las pruebas históricas de OpenAI están archivadas como dev1. La cobertura de palabras distingue invocación de factories, ciclo simulado y disponibilidad real de infraestructura.